**Last code edit:** 2026-08-16 14:56 (UTC+03:00)

# Module 4 — Recommender Systems

This notebook implements two recommenders for implicit listening data:

1. **Content-based:** represent each user as a listen-weighted aggregate of the artist–tag TF-IDF vectors created in Module 3.
2. **Collaborative filtering:** recommend artists preferred by users with similar listening histories.

The TF-IDF matrix is useful because it gives every tagged artist a position in a shared semantic space. A user's profile is the weighted centroid of the artists they listened to. Cosine similarity then retrieves unlistened artists whose tags point in the same direction. Common tags receive less influence through IDF, while more distinctive tags carry more information.

In [ ]:
# Where the cleaned tables and the Module 3 artifacts come from.
#
# On a laptop everything is already in the project folder. Colab gives every
# notebook its own temporary virtual machine, so there the inputs are fetched
# either from the GitHub repository, which carries the committed copies, or from
# MyDrive/bi130_colab/, where modules 0 and 3 publish theirs. Colab renders
# DATA_SOURCE as a dropdown, so the two options exclude each other.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

DATA_SOURCE = "github"  # @param ["github", "google_drive"]

REPO_URL = "https://github.com/Areso/music-recommendation-system.git"
REPO_BRANCH = "master"
COLAB_REPO_DIR = Path("/content/music-recommendation-system")
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "bi130_colab"


def clone_repo():
    """Clone the project repository, or refresh an existing clone, and return it."""
    if COLAB_REPO_DIR.is_dir():
        subprocess.run(
            ["git", "-C", str(COLAB_REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(COLAB_REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
             REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    return COLAB_REPO_DIR


def mount_drive():
    """Mount Google Drive and return the folder shared between the modules."""
    from google.colab import drive

    if not (DRIVE_MOUNT_POINT / "MyDrive").is_dir():
        drive.mount(str(DRIVE_MOUNT_POINT))
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    return DRIVE_PROJECT_DIR


DATA_SOURCES = {"github": clone_repo, "google_drive": mount_drive}

if IN_COLAB:
    if DATA_SOURCE not in DATA_SOURCES:
        raise ValueError(
            f"DATA_SOURCE must be one of {sorted(DATA_SOURCES)}, got {DATA_SOURCE!r}"
        )
    os.chdir(DATA_SOURCES[DATA_SOURCE]())

# Module 0 supplies clean/, Module 3 supplies the three TF-IDF artifacts.
REQUIRED_INPUTS = (
    "clean",
    "tfidf_matrix.npz",
    "vectorizer.joblib",
    "artist_ids.json",
)
missing_inputs = [name for name in REQUIRED_INPUTS if not (Path.cwd() / name).exists()]
if missing_inputs:
    raise FileNotFoundError(
        f"Missing in {Path.cwd()}: {missing_inputs}. Run modules 0 and 3 first, and "
        "when reading from Drive run them with STORE_TO_DRIVE ticked."
    )

print(f"IN_COLAB:    {IN_COLAB}")
print(f"Data source: {DATA_SOURCE if IN_COLAB else 'local project folder'}")
print(f"Project dir: {Path.cwd()}")

In [1]:
from pathlib import Path
import __main__
import json
import joblib

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.sparse import csr_matrix, load_npz
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
TOP_N = 10
N_NEIGHBORS = 30

DATA_DIR = Path("clean")

# Listening history and display names.
listens = pd.read_csv(
    DATA_DIR / "user_artists_clean.csv",
    dtype={"userID": "int64", "artistID": "int64", "weight": "float64"},
)
artists = pd.read_csv(DATA_DIR / "artists_clean.csv")
artist_name = (
    artists.set_index("artistID")["source_name_repaired"]
    .fillna(artists.set_index("artistID")["canonical_name"])
    .to_dict()
)

# Module 3 artifacts. The tokenizer must exist in __main__ to unpickle the vectorizer.
def newline_tokenizer(text):
    return [line.strip() for line in text.splitlines() if line.strip()]


__main__.newline_tokenizer = newline_tokenizer
with open("artist_ids.json", encoding="utf-8") as file:
    content_artist_ids = np.asarray(json.load(file), dtype=np.int64)

tfidf_matrix = load_npz("tfidf_matrix.npz").tocsr()
vectorizer = joblib.load("vectorizer.joblib")
feature_names = vectorizer.get_feature_names_out()
content_row_of = {artist_id: row for row, artist_id in enumerate(content_artist_ids)}

assert tfidf_matrix.shape == (len(content_artist_ids), len(feature_names))
assert len(content_row_of) == len(content_artist_ids)
assert not listens.duplicated(["userID", "artistID"]).any()

print(f"Listening data: {listens['userID'].nunique():,} users × {listens['artistID'].nunique():,} artists")
print(f"Content matrix: {tfidf_matrix.shape[0]:,} artists × {tfidf_matrix.shape[1]:,} tags")

Listening data: 1,892 users × 17,619 artists
Content matrix: 11,841 artists × 3,515 tags


## Implicit-feedback confidence

A listen is not an explicit rating: it indicates interaction strength, not necessarily a 1–5 preference. This notebook uses

\[
c_{ua}=\log(1+\mathrm{listens}_{ua})
\]

as confidence for both recommenders. Raw counts are extremely skewed and would let a few artists dominate; binarizing would discard the useful distinction between an artist played once and one played thousands of times. `log1p` keeps that distinction while compressing the extremes.

In [2]:
listens = listens.assign(
    confidence=np.log1p(listens["weight"]),
    has_content_vector=listens["artistID"].isin(content_row_of),
)

user_coverage = (
    listens.groupby("userID")
    .agg(
        listened_artists=("artistID", "size"),
        content_artists=("has_content_vector", "sum"),
        total_listens=("weight", "sum"),
    )
)
user_coverage["content_artist_share"] = (
    user_coverage["content_artists"] / user_coverage["listened_artists"]
)

coverage_summary = pd.DataFrame(
    {
        "measure": [
            "Artists in listening data",
            "Artists with TF-IDF content vectors",
            "Listening rows covered by TF-IDF",
            "Listen weight covered by TF-IDF",
        ],
        "value": [
            f"{listens['artistID'].nunique():,}",
            f"{len(content_artist_ids):,}",
            f"{listens['has_content_vector'].mean():.1%}",
            f"{listens.loc[listens['has_content_vector'], 'weight'].sum() / listens['weight'].sum():.1%}",
        ],
    }
)
display(coverage_summary)

auto_candidates = user_coverage.query("content_artists >= 5").index.difference([2]).to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
SAMPLE_USERS = [2, *sorted(rng.choice(auto_candidates, size=2, replace=False).tolist())]
print("Reproducible sample users:", SAMPLE_USERS)

,measure,value
0,Artists in listening data,"17,619"
1,Artists with TF-IDF content vectors,"11,841"
2,Listening rows covered by TF-IDF,92.9%
3,Listen weight covered by TF-IDF,96.9%


Reproducible sample users: [2, 179, 1622]


## 1. Content-based recommender

For user \(u\), let \(v_a\) be artist \(a\)'s TF-IDF tag vector. The user profile is the normalized listen-weighted aggregate

\[
p_u = \operatorname{normalize}\left(\frac{\sum_{a\in L_u} c_{ua}v_a}{\sum_{a\in L_u}c_{ua}}\right).
\]

Artists without a Module 3 vector cannot contribute to the profile or be content candidates. They are still excluded if the user has already listened to them.

In [3]:
def top_features(vector, n=5):
    """Return the strongest tag names in one sparse vector."""
    row = csr_matrix(vector)
    if row.nnz == 0:
        return []
    order = np.argsort(row.data)[::-1][:n]
    return feature_names[row.indices[order]].tolist()


def build_user_content_profile(user_id):
    """Build one normalized user profile from available Module 3 artist vectors."""
    history = listens.loc[listens["userID"].eq(user_id)]
    if history.empty:
        raise ValueError(f"Unknown userID: {user_id}")

    covered = history.loc[history["has_content_vector"]]
    if covered.empty:
        raise ValueError(f"User {user_id} has no listened artists with content vectors")

    rows = np.fromiter(
        (content_row_of[artist_id] for artist_id in covered["artistID"]),
        dtype=np.int64,
    )
    confidence = covered["confidence"].to_numpy()
    weighted_sum = tfidf_matrix[rows].multiply(confidence[:, None]).sum(axis=0)
    profile = csr_matrix(weighted_sum / confidence.sum())

    norm = np.sqrt(profile.multiply(profile).sum())
    if norm > 0:
        profile = profile / norm
    return profile, len(covered), len(history)


def recommend_content(user_id, n=TOP_N):
    """Recommend unlistened artists nearest to the user's tag profile."""
    profile, covered_count, history_count = build_user_content_profile(user_id)
    scores = cosine_similarity(profile, tfidf_matrix).ravel()

    seen_artist_ids = set(listens.loc[listens["userID"].eq(user_id), "artistID"])
    seen_rows = [content_row_of[artist_id] for artist_id in seen_artist_ids if artist_id in content_row_of]
    scores[seen_rows] = -np.inf

    candidate_rows = np.flatnonzero(np.isfinite(scores))
    ranked_rows = candidate_rows[np.argsort(scores[candidate_rows])[::-1][:n]]

    result = pd.DataFrame(
        {
            "artistID": content_artist_ids[ranked_rows],
            "artist": [artist_name.get(artist_id, f"Artist {artist_id}") for artist_id in content_artist_ids[ranked_rows]],
            "content_score": scores[ranked_rows],
            "top_artist_tags": [", ".join(top_features(tfidf_matrix.getrow(row))) for row in ranked_rows],
        }
    )
    result.attrs["profile_tags"] = top_features(profile, n=10)
    result.attrs["content_coverage"] = covered_count / history_count
    return result

In [4]:
def listening_history(user_id, n=10):
    history = (
        listens.loc[listens["userID"].eq(user_id), ["artistID", "weight", "confidence"]]
        .sort_values("weight", ascending=False)
        .head(n)
        .copy()
    )
    history.insert(1, "artist", history["artistID"].map(artist_name))
    return history


user_id = 2
content_example = recommend_content(user_id)
display(Markdown(f"### Content example — user {user_id}"))
display(Markdown("**Strongest profile tags:** " + ", ".join(content_example.attrs["profile_tags"])))
display(Markdown(f"**Content coverage:** {content_example.attrs['content_coverage']:.0%} of listened artists"))
display(listening_history(user_id))
display(content_example)

### Content example — user 2

**Strongest profile tags:** electronic, 1980s, new wave, trip hop, chill out, dance, pop, synth-pop, female vocalist, alternative

**Content coverage:** 98% of listened artists

,artistID,artist,weight,confidence
0,51,Duran Duran,13883.0,9.538492
1,52,Morcheeba,11690.0,9.366575
2,53,Air,11351.0,9.337149
3,54,Hooverphonic,10300.0,9.239996
4,55,Kylie Minogue,8983.0,9.103200
5,56,Daft Punk,6152.0,8.724695
6,57,Thievery Corporation,5955.0,8.692154
7,58,Goldfrapp,4616.0,8.437500
8,59,New Order,4337.0,8.375169
9,60,Matt Bianco,4147.0,8.330382


,artistID,artist,content_score,top_artist_tags
0,1001,Pet Shop Boys,0.715773,"synth-pop, 1980s, new wave, electronic, pop"
1,1892,Eurythmics,0.690392,"new wave, 1980s, synth-pop, pop, female vocalist"
2,11243,Freur,0.676252,"1980s, new romantic, electronic, atmospheric, ..."
3,1004,Kim Wilde,0.668806,"new wave, 1980s, pop, female vocalist, british"
4,18599,The Pet Shop Boys,0.661736,"synth-pop, new wave, british, 1980s, electronic"
5,3768,Soft Cell,0.654195,"new wave, synth-pop, 1980s, new romantic, elec..."
6,193,Tears for Fears,0.652356,"1980s, new wave, synth-pop, british, pop"
7,13426,R.O. Manse,0.652253,"new romantic, synth-pop, new wave, british, 1980s"
8,187,a-ha,0.647266,"new wave, 1980s, synth-pop, pop, norwegian"
9,998,Orchestral Manoeuvres in the Dark,0.645706,"new wave, synth-pop, 1980s, electronic, new ro..."


## 2. User-based neighborhood collaborative filtering

The rows of the sparse user–artist matrix contain `log1p` listening confidence. Cosine similarity compares the direction of two listening profiles, so users can be similar even when their total listening volume differs.

For a target user, the recommender selects the most similar positive-similarity neighbors and computes a similarity-weighted sum of their artist confidence. It excludes the target user's history. This learns co-listening patterns directly and can score all 17,619 artists in the listening data, including artists without tags.

In [5]:
user_ids = np.sort(listens["userID"].unique())
cf_artist_ids = np.sort(listens["artistID"].unique())
user_row_of = {user_id: row for row, user_id in enumerate(user_ids)}
cf_col_of = {artist_id: col for col, artist_id in enumerate(cf_artist_ids)}

user_rows = listens["userID"].map(user_row_of).to_numpy()
artist_cols = listens["artistID"].map(cf_col_of).to_numpy()
user_artist_matrix = csr_matrix(
    (listens["confidence"].to_numpy(), (user_rows, artist_cols)),
    shape=(len(user_ids), len(cf_artist_ids)),
)

assert user_artist_matrix.nnz == len(listens)
print(
    f"CF matrix: {user_artist_matrix.shape[0]:,} users × "
    f"{user_artist_matrix.shape[1]:,} artists; "
    f"density={user_artist_matrix.nnz / np.prod(user_artist_matrix.shape):.3%}"
)


def recommend_user_cf(user_id, n=TOP_N, n_neighbors=N_NEIGHBORS):
    """Recommend from the similarity-weighted confidence of nearest users."""
    if user_id not in user_row_of:
        raise ValueError(f"Unknown userID: {user_id}")

    user_row = user_row_of[user_id]
    similarities = cosine_similarity(
        user_artist_matrix.getrow(user_row), user_artist_matrix
    ).ravel()
    similarities[user_row] = 0.0

    ranked_users = np.argsort(similarities)[::-1]
    neighbor_rows = ranked_users[similarities[ranked_users] > 0][:n_neighbors]
    if len(neighbor_rows) == 0:
        return pd.DataFrame(
            columns=["artistID", "artist", "cf_score", "neighbor_support"]
        )

    neighbor_matrix = user_artist_matrix[neighbor_rows]
    neighbor_similarities = similarities[neighbor_rows]
    scores = np.asarray(
        neighbor_matrix.multiply(neighbor_similarities[:, None]).sum(axis=0)
    ).ravel() / neighbor_similarities.sum()
    support = np.asarray(neighbor_matrix.getnnz(axis=0)).ravel()

    seen_cols = user_artist_matrix.getrow(user_row).indices
    scores[seen_cols] = -np.inf
    candidate_cols = np.flatnonzero(np.isfinite(scores) & (scores > 0))
    ranked_cols = candidate_cols[np.argsort(scores[candidate_cols])[::-1][:n]]

    result = pd.DataFrame(
        {
            "artistID": cf_artist_ids[ranked_cols],
            "artist": [artist_name.get(artist_id, f"Artist {artist_id}") for artist_id in cf_artist_ids[ranked_cols]],
            "cf_score": scores[ranked_cols],
            "neighbor_support": support[ranked_cols],
        }
    )
    result.attrs["neighbors_used"] = len(neighbor_rows)
    result.attrs["top_neighbor_similarity"] = similarities[neighbor_rows[0]]
    return result

CF matrix: 1,892 users × 17,619 artists; density=0.278%


In [6]:
cf_example = recommend_user_cf(user_id)
display(Markdown(f"### Collaborative-filtering example — user {user_id}"))
display(
    Markdown(
        f"Using **{cf_example.attrs['neighbors_used']}** neighbors; "
        f"closest-neighbor cosine similarity = **{cf_example.attrs['top_neighbor_similarity']:.3f}**."
    )
)
display(cf_example)

### Collaborative-filtering example — user 2

Using **30** neighbors; closest-neighbor cosine similarity = **0.260**.

,artistID,artist,cf_score,neighbor_support
0,1001,Pet Shop Boys,3.480882,16
1,187,a-ha,2.814800,15
2,1014,Erasure,2.685715,14
3,159,The Cure,2.603759,14
4,154,Radiohead,2.381783,12
5,993,Simple Minds,2.181265,11
6,1892,Eurythmics,2.177695,13
7,511,U2,2.008520,10
8,193,Tears for Fears,2.004656,12
9,599,David Bowie,1.833223,10


## Sample top-N recommendations

User 2 was shown above in detail. The next cell runs both recommenders for two additional reproducibly selected users. Their outputs are not expected to match: content similarity follows tag semantics, while CF follows collective listening behavior.

In [7]:
for sample_user_id in SAMPLE_USERS[1:]:
    display(Markdown(f"### User {sample_user_id}"))
    display(Markdown("**Most-listened artists**"))
    display(listening_history(sample_user_id, n=5))
    display(Markdown("**Content-based top 10**"))
    display(recommend_content(sample_user_id))
    display(Markdown("**User-CF top 10**"))
    display(recommend_user_cf(sample_user_id))

### User 179

**Most-listened artists**

,artistID,artist,weight,confidence
8368,89,Lady Gaga,4263.0,8.357963
8376,289,Britney Spears,3853.0,8.256867
8385,377,Linkin Park,2271.0,7.728416
8408,2432,Porcelain and the Tramps,2184.0,7.689371
8396,523,Lindsay Lohan,1192.0,7.084226


**Content-based top 10**

,artistID,artist,content_score,top_artist_tags
0,525,Gwen Stefani,0.749238,"pop, dance, female vocalist, gwen stefani, pop..."
1,534,No Doubt,0.742715,"alternative, pop, rock, 1990s, female vocalist"
2,316,Alanis Morissette,0.736982,"female vocalist, rock, alternative, singer-son..."
3,302,P!nk,0.736702,"pop rock, pop, female vocalist, rock, dance"
4,538,Maroon 5,0.679971,"pop rock, pop, rock, alternative rock, alterna..."
5,967,Sixpence None the Richer,0.672746,"pop, alternative, rock, pop rock, female vocalist"
6,962,The Cardigans,0.670648,"female vocalist, swedish, pop, alternative, rock"
7,972,t.A.T.u.,0.668621,"pop, female vocalist, electronic, dance, russian"
8,18460,Discobitch,0.656465,"dance, female vocalist, pop, electronic"
9,1687,Liz Phair,0.653554,"female vocalist, alternative rock, underrated,..."


...

,artistID,artist,weight,confidence
71841,1837,Scott Walker,644.0,6.469250
71837,874,Roxy Music,616.0,6.424869
71862,10105,The Sound,579.0,6.363028
71833,599,David Bowie,432.0,6.070738
71871,15637,The Teardrop Explodes,402.0,5.998937


**Content-based top 10**

,artistID,artist,content_score,top_artist_tags
0,6883,Modern English,0.685274,"post-punk, new wave, british, 1980s, synth-pop"
1,7076,Gang of Four,0.665245,"post-punk, new wave, 1980s, lms artist, first-..."
2,3292,The Glove,0.661922,"post-punk, new wave, the cure, gothic, british"
3,159,The Cure,0.651837,"new wave, post-punk, 1980s, alternative, gothic"
4,861,Killing Joke,0.650845,"post-punk, industrial, new wave, 1980s, punk"
5,3451,Love and Rockets,0.649538,"post-punk, balearic, new wave, 1980s, gothic"
6,1083,Siouxsie and the Banshees,0.635076,"post-punk, gothic, new wave, 1980s, punk"
7,2488,Public Image Ltd.,0.634795,"post-punk, 1980s, 9100, choon, alternative"
8,5912,Sad Lovers and Giants,0.634497,"post-punk, new wave, gothic, 1980s"
9,1017,The Psychedelic Furs,0.626935,"new wave, post-punk, 1980s, alternative, colle..."


**User-CF top 10**

,artistID,artist,cf_score,neighbor_support
0,227,The Beatles,4.426097,19
1,159,The Cure,3.555138,17
2,72,Depeche Mode,3.233972,14
3,735,The Rolling Stones,3.141852,17
4,868,The Doors,2.842379,14
5,163,Pink Floyd,2.819007,14
6,429,The Smiths,2.316890,12
7,59,New Order,2.207076,11
8,1412,Led Zeppelin,2.150004,11
9,154,Radiohead,1.961837,10


In [8]:
artist_listener_count = listens.groupby("artistID")["userID"].nunique()
comparison_rows = []

for sample_user_id in SAMPLE_USERS:
    content_recs = recommend_content(sample_user_id)
    cf_recs = recommend_user_cf(sample_user_id)
    content_ids = set(content_recs["artistID"])
    cf_ids = set(cf_recs["artistID"])
    comparison_rows.append(
        {
            "userID": sample_user_id,
            "content_history_coverage": content_recs.attrs["content_coverage"],
            "top_10_overlap": len(content_ids & cf_ids),
            "content_median_listeners": content_recs["artistID"].map(artist_listener_count).median(),
            "cf_median_listeners": cf_recs["artistID"].map(artist_listener_count).median(),
            "CF_neighbors": cf_recs.attrs["neighbors_used"],
        }
    )

comparison = pd.DataFrame(comparison_rows)
display(comparison.style.format({"content_history_coverage": "{:.0%}"}))

,userID,content_history_coverage,top_10_overlap,content_median_listeners,cf_median_listeners,CF_neighbors
0,2,98%,4,28.500000,95.000000,30
1,179,100%,1,83.000000,273.500000,30
2,1622,98%,1,6.500000,212.500000,30


## Short comparison

**Content-based is stronger when:**
- a user has only a small history but at least one well-tagged artist;
- recommendations need a human-readable explanation (profile tags and artist tags);
- niche musical properties matter more than overall popularity;
- a new artist has descriptive tags but too little interaction history for CF.

Its main weaknesses here are the content gap (5,778 listened artists have no TF-IDF row), dependence on tag quality, and **over-specialization**: it tends to return more of the same tag profile.

**Collaborative filtering is stronger when:**
- many users have overlapping histories, allowing behavior to reveal relationships that tags miss;
- useful recommendations cross genre/tag boundaries (greater serendipity);
- an artist has listening activity but incomplete or no tags.

Its main weaknesses are user cold-start, weak similarity for unusual users, and popularity bias because frequently listened-to artists receive support from more neighbors. The `top_10_overlap` and median-listener columns above make those differences visible for the sample users. In practice, a hybrid can combine normalized content and CF scores to get both semantic relevance and behavioral discovery.

In [9]:
# Smoke checks for the demonstrated users.
for sample_user_id in SAMPLE_USERS:
    seen = set(listens.loc[listens["userID"].eq(sample_user_id), "artistID"])
    content_recs = recommend_content(sample_user_id, n=TOP_N)
    cf_recs = recommend_user_cf(sample_user_id, n=TOP_N)

    for label, recommendations, score_column in [
        ("content", content_recs, "content_score"),
        ("CF", cf_recs, "cf_score"),
    ]:
        assert len(recommendations) == TOP_N, f"{label}: expected {TOP_N} rows"
        assert recommendations["artistID"].is_unique
        assert seen.isdisjoint(recommendations["artistID"])
        assert np.isfinite(recommendations[score_column]).all()
        assert recommendations[score_column].is_monotonic_decreasing

print(f"All recommendation checks passed for users {SAMPLE_USERS}.")

All recommendation checks passed for users [2, 179, 1622].


## Export collaborative-filtering artifacts

The API needs the exact sparse confidence matrix and both axis mappings. Saving them here avoids rebuilding the matrix from CSV every time the server starts:

- `user_artist_matrix.npz`: CSR matrix of `log1p(weight)` confidence;
- `cf_user_ids.json`: matrix row → user ID;
- `cf_artist_ids.json`: matrix column → artist ID.

These three files must always be exported together so their ordering remains aligned.

In [10]:
from scipy.sparse import save_npz

CF_MATRIX_PATH = Path("user_artist_matrix.npz")
CF_USER_IDS_PATH = Path("cf_user_ids.json")
CF_ARTIST_IDS_PATH = Path("cf_artist_ids.json")

save_npz(CF_MATRIX_PATH, user_artist_matrix)
CF_USER_IDS_PATH.write_text(json.dumps(user_ids.tolist()), encoding="utf-8")
CF_ARTIST_IDS_PATH.write_text(json.dumps(cf_artist_ids.tolist()), encoding="utf-8")

# Verify that the serialized mappings still describe the matrix axes.
exported_user_ids = json.loads(CF_USER_IDS_PATH.read_text(encoding="utf-8"))
exported_artist_ids = json.loads(CF_ARTIST_IDS_PATH.read_text(encoding="utf-8"))
assert user_artist_matrix.shape == (len(exported_user_ids), len(exported_artist_ids))

print("Exported CF artifacts:")
print(f"- {CF_MATRIX_PATH}: {user_artist_matrix.shape}, {user_artist_matrix.nnz:,} non-zero values")
print(f"- {CF_USER_IDS_PATH}: {len(exported_user_ids):,} user IDs")
print(f"- {CF_ARTIST_IDS_PATH}: {len(exported_artist_ids):,} artist IDs")

Exported CF artifacts:
- user_artist_matrix.npz: (1892, 17619), 92,829 non-zero values
- cf_user_ids.json: 1,892 user IDs
- cf_artist_ids.json: 17,619 artist IDs
